# Week 2 - Module 2: Model Stability (PyTorch)
### He Initialization + BatchNorm + Dropout (0.3) - Ablation vs plain baseline

Your Week 1 `ClinicalMLP` already had He-init/BatchNorm/Dropout folded in. This notebook splits that apart so Module 2 is a real, separate, measurable deliverable: a **plain baseline MLP** (LeakyReLU only, default PyTorch init, no BatchNorm, no Dropout) vs the **stabilized MLP** (your existing `ClinicalMLP`), trained on the identical data/split/seed.

Run all cells top to bottom.

In [ ]:
import os
import sys
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/Cardiac_Risk_Prediction'
except ImportError:
    PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())

os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/outputs', exist_ok=True)
if f'{PROJECT_DIR}/src' not in sys.path:
    sys.path.append(f'{PROJECT_DIR}/src')

print(f'Working directory permanently mapped to: {PROJECT_DIR}')


## Data (identical generator/seed/split to Week 1, so the ablation is fair)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Import shared generator
from data_utils import load_clinical_metadata

X, y = load_clinical_metadata(n_samples=2000)
X_train_df, X_val_df, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_df).astype(np.float32)
X_val = scaler.transform(X_val_df).astype(np.float32)

train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train).unsqueeze(1))
val_dataset = TensorDataset(torch.tensor(X_val), torch.tensor(y_val).unsqueeze(1))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')


## Two models
- **BaselineMLP** - plain `Linear -> LeakyReLU`, PyTorch's default init, no BatchNorm, no Dropout. This is what Module 1 would be *without* any Module 2 additions.
- **ClinicalMLP** - your existing Week 1 architecture: He-init (`kaiming_normal_`) + BatchNorm + Dropout(0.3), unchanged from your Week 1 notebook.

In [ ]:
# Import both models from src/models.py to eliminate code drift
from models import BaselineMLP, ClinicalMLP
print('BaselineMLP and ClinicalMLP imported from src/models.py.')


## Shared training loop (used for both models so the comparison is fair)

In [ ]:
def train_model(model, checkpoint_path, epochs=50, seed=42):
    torch.manual_seed(seed)
    model = model.to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    best_val_auc = 0.0
    history = {'train_loss': [], 'val_loss': [], 'val_auc': []}

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0.0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            preds = model(batch_x)
            loss = criterion(preds, batch_y)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
        avg_train_loss = total_train_loss / len(train_loader)

        model.eval()
        total_val_loss = 0.0
        val_preds, val_targets = [], []
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                preds = model(batch_x)
                loss = criterion(preds, batch_y)
                total_val_loss += loss.item()
                val_preds.extend(preds.cpu().numpy())
                val_targets.extend(batch_y.cpu().numpy())
        avg_val_loss = total_val_loss / len(val_loader)
        current_val_auc = roc_auc_score(val_targets, val_preds)

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_auc'].append(current_val_auc)

        if current_val_auc > best_val_auc:
            best_val_auc = current_val_auc
            torch.save({'model_state_dict': model.state_dict(), 'val_auc': best_val_auc}, checkpoint_path)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1:02d}/{epochs}] | Train Loss: {avg_train_loss:.4f} | "
                  f"Val Loss: {avg_val_loss:.4f} | Val AUC: {current_val_auc:.4f}")

    return model, history, best_val_auc


## Run the ablation: BaselineMLP vs ClinicalMLP (Module 2 stabilized)

In [ ]:
print('Training BaselineMLP (no He-init / BatchNorm / Dropout)...')
baseline_model, hist_base, best_auc_base = train_model(
    BaselineMLP(input_dim=X_train.shape[1]),
    checkpoint_path=f'{PROJECT_DIR}/checkpoints/module1_baseline_plain.pt',
    epochs=50
)

print('\nTraining ClinicalMLP (He-init + BatchNorm + Dropout 0.3)...')
stable_model, hist_stable, best_auc_stable = train_model(
    ClinicalMLP(input_dim=X_train.shape[1]),
    checkpoint_path=f'{PROJECT_DIR}/checkpoints/module2_stable_mlp.pt',
    epochs=50
)


## Compare results

In [ ]:
results = pd.DataFrame([
    {'model': 'Module 1 (plain baseline)', 'best_val_auc': round(best_auc_base, 4)},
    {'model': 'Module 2 (He-init + BatchNorm + Dropout 0.3)', 'best_val_auc': round(best_auc_stable, 4)},
])
results.to_csv(f'{PROJECT_DIR}/outputs/week2_ablation_results.csv', index=False)
print(results.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(hist_base['val_auc'], label='Module 1 (baseline)')
axes[0].plot(hist_stable['val_auc'], label='Module 2 (stabilized)')
axes[0].set_title('Validation AUC')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('AUC'); axes[0].legend()

axes[1].plot(hist_base['val_loss'], label='Module 1 (baseline)')
axes[1].plot(hist_stable['val_loss'], label='Module 2 (stabilized)')
axes[1].set_title('Validation Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].legend()

plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/outputs/week2_ablation_auc_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved plot and checkpoints to Drive.')


## Interpretation (fill in after your run)
- Compare `best_val_auc` between the two rows above.
- Watch the **loss curves**, not just the final AUC: the stabilized model's val loss should be visibly smoother / less spiky epoch-to-epoch than the plain baseline's - that smoothness is the actual signature of BatchNorm + Dropout + He-init doing their job (stable gradient flow), separate from whatever the AUC number says.
- If AUC is similar between the two on this dataset, that's a legitimate and reportable result: this synthetic clinical dataset is small/low-dimensional, so regularization benefits will show up more clearly once Module 3's deeper 1D-CNN is added.